[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/projects/06_nlp/project.ipynb)

# Project 06 — Phân loại chủ đề tin tức tiếng Việt (NLP)

> Một trang tin cần tự động gắn **chủ đề** (kinh tế, giáo dục, thể thao, sức khỏe, du lịch, công nghệ) cho bài viết mới để đưa vào đúng chuyên mục. Bạn có 3.000 tin đã gắn nhãn thủ công (có ~3% nhãn sai — biên tập viên cũng nhầm), và cần một mô hình nhanh, chính xác, hiểu được.

**Bạn sẽ làm**:
1. EDA: phân bố chủ đề, độ dài tin.
2. Baseline: lớp đa số, Naive Bayes trên đếm từ.
3. TF-IDF + hồi quy logistic: so sánh **âm tiết đơn**, **âm tiết + cặp âm tiết** (vì từ tiếng Việt thường gồm nhiều âm tiết), **n-gram ký tự** bằng cross-validation.
4. Đánh giá test: macro-F1, ma trận nhầm lẫn, phân tích lỗi; đường cong học; độ bền với tin nhiều câu gây nhiễu.
5. (Mở rộng, cần PyTorch) mô hình embedding kiểu fastText và **Transformer encoder nhỏ** học từ đầu; báo cáo.

**Tiêu chí đạt**: CV chọn cấu hình có cặp âm tiết · macro-F1 test > 0.94 · cặp âm tiết tốt hơn âm tiết đơn trên test · macro-F1 trên bộ tin khó > 0.9.

**Chương liên quan**: 09 (Naive Bayes văn bản), 07 (logistic), 32 (tiền xử lý, BoW/TF-IDF, embedding), 31 (Transformer), 14–15 (metric, CV), 05 (đường cong học).

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/projects/06_nlp

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_news_vi

train = make_news_vi(3000, seed=0)
test = make_news_vi(1000, seed=1)
hard = make_news_vi(1000, seed=2, distractor=0.6, label_noise=0.0)   # tin có nhiều câu không liên quan tới chủ đề
print(train.shape, test.shape)
train.head()

## Bước 1 — EDA

Tính `class_share` (tỉ lệ mỗi chủ đề trong train), `length_stats` (mô tả số âm tiết mỗi tin — `describe()` của số token tách bằng khoảng trắng). Lớp có cân bằng không?

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
class_share = train.topic.value_counts(normalize=True)
length_stats = train.text.str.split().str.len().describe()
### END CODE HERE ###
print(class_share.round(3).to_dict())
print(length_stats.round(1).to_dict())

In [ ]:
check_equal(len(class_share), 6, "6 chủ đề")
check_less(class_share.max() / class_share.min(), 1.3, "các lớp gần cân bằng → accuracy ≈ macro-F1")

## Bước 2 — Baseline

`f1_majority`: macro-F1 khi luôn dự đoán chủ đề nhiều nhất của train. `nb`: `CountVectorizer(ngram_range=(1, 2))` + `MultinomialNB()` → `f1_nb` trên test.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
f1_majority = f1_score(test.topic, np.full(len(test), train.topic.mode()[0]), average="macro")
nb = make_pipeline(CountVectorizer(ngram_range=(1, 2)), MultinomialNB()).fit(train.text, train.topic)
f1_nb = f1_score(test.topic, nb.predict(test.text), average="macro")
### END CODE HERE ###
print(f"macro-F1 — lớp đa số: {f1_majority:.3f} | Naive Bayes: {f1_nb:.3f}")

In [ ]:
check_less(f1_majority, 0.1, "lớp đa số vô dụng với macro-F1")
check_greater(f1_nb, 0.9, "Naive Bayes đã rất mạnh cho phân loại chủ đề")

## Bước 3 — TF-IDF + logistic: chọn biểu diễn bằng CV

Ba pipeline (đều `LogisticRegression(max_iter=2000, C=10)`):
- `"âm tiết"`: `TfidfVectorizer(ngram_range=(1, 1))`
- `"âm tiết + cặp"`: `TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)`
- `"ký tự 2–5"`: `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)`

5-fold CV phân tầng (macro-F1) trên train → `cv_scores` (Series); `best_name` = cấu hình tốt nhất.

In [ ]:
pipelines = {
    "âm tiết": make_pipeline(TfidfVectorizer(ngram_range=(1, 1)), LogisticRegression(max_iter=2000, C=10)),
    "âm tiết + cặp": make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=2000, C=10)),
    "ký tự 2–5": make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2), LogisticRegression(max_iter=2000, C=10)),
}
### START CODE HERE ### (≈ 3 dòng)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
cv_scores = pd.Series({name: cross_val_score(p, train.text, train.topic, cv=cv, scoring="f1_macro").mean() for name, p in pipelines.items()})
best_name = cv_scores.idxmax()
### END CODE HERE ###
print(cv_scores.round(4).to_string(), "\n→ chọn:", best_name)

In [ ]:
check_equal(best_name, "âm tiết + cặp", "CV chọn âm tiết + cặp âm tiết")
check_greater(cv_scores["âm tiết + cặp"], cv_scores["âm tiết"] + 0.01, "cặp âm tiết giúp rõ rệt")

Tiếng Việt viết tách **âm tiết**, không tách **từ**: "bóng đá", "chứng khoán", "du lịch" là một từ hai âm tiết. Âm tiết đơn "bóng", "chứng", "du" mơ hồ; cặp âm tiết khôi phục phần lớn từ ghép mà không cần bộ tách từ (word segmenter như VnCoreNLP, underthesea — có thể thử thêm).

## Bước 4 — Đánh giá test và phân tích lỗi

Huấn luyện `best = pipelines[best_name]` trên train → `pred_test`; `f1_test`; F1 của cấu hình âm tiết đơn trên test `f1_unigram_test`; vẽ ma trận nhầm lẫn; lấy DataFrame `errors` (cột `text`, `topic`, `predicted`) gồm các tin dự đoán sai.

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
best = pipelines[best_name].fit(train.text, train.topic)
pred_test = best.predict(test.text)
f1_test = f1_score(test.topic, pred_test, average="macro")
f1_unigram_test = f1_score(test.topic, pipelines["âm tiết"].fit(train.text, train.topic).predict(test.text), average="macro")
errors = test.assign(predicted=pred_test).loc[lambda d: d.topic != d.predicted, ["text", "topic", "predicted"]]
### END CODE HERE ###
print(f"macro-F1 test: {f1_test:.3f} (âm tiết đơn: {f1_unigram_test:.3f}) | accuracy {accuracy_score(test.topic, pred_test):.3f} | {len(errors)} tin sai")
ConfusionMatrixDisplay.from_predictions(test.topic, pred_test, xticks_rotation=45, cmap="Blues")
plt.show()
pd.set_option("display.max_colwidth", 120)
errors.head(8)

In [ ]:
check_greater(f1_test, 0.94, "macro-F1 test > 0.94")
check_greater(f1_test, f1_unigram_test, "cặp âm tiết tốt hơn âm tiết đơn trên test")
check_equal(len(errors), int((pred_test != test.topic.values).sum()), "bảng lỗi đầy đủ")

**Phân tích lỗi mẫu:** Đọc các tin sai, phần lớn rơi vào hai nhóm: (1) **nhãn sai** — nội dung rõ ràng thuộc chủ đề mô hình dự đoán (dữ liệu có ~3% nhãn nhiễu; mô hình đúng mà "bị chấm sai") → trần macro-F1 thực tế ~0.97; (2) tin có nhiều câu **gây nhiễu** thuộc chủ đề khác (ví dụ tin du lịch nhắc tới chi phí, tỉ giá → kinh tế). Hành động: gửi các tin mô hình rất tự tin nhưng trái nhãn cho biên tập viên rà soát (sửa nhãn thường hiệu quả hơn đổi mô hình); với tin đa chủ đề cân nhắc **đa nhãn**.

## Bước 5 — Đường cong học và độ bền

`learning_curve_f1`: macro-F1 trên test khi huấn luyện cấu hình tốt nhất trên 100, 300, 1000, 3000 tin đầu của train (Series). `f1_hard`: macro-F1 của `best` trên bộ tin khó.

In [ ]:
from sklearn.base import clone

### START CODE HERE ### (≈ 3 dòng)
learning_curve_f1 = pd.Series({n: f1_score(test.topic, clone(pipelines[best_name]).fit(train.text[:n], train.topic[:n]).predict(test.text), average="macro")
                               for n in [100, 300, 1000, 3000]})
f1_hard = f1_score(hard.topic, best.predict(hard.text), average="macro")
### END CODE HERE ###
print("đường cong học:", learning_curve_f1.round(3).to_dict(), f"| bộ tin khó: {f1_hard:.3f}")

In [ ]:
check_true(learning_curve_f1.is_monotonic_increasing, "thêm dữ liệu → tốt hơn")
check_greater(learning_curve_f1[3000] - learning_curve_f1[1000], 0.0, "vẫn còn cải thiện ở 3000 tin")
check_greater(f1_hard, 0.9, "bền với tin nhiều câu gây nhiễu")

## Bước 6 — Diễn giải

Lấy 5 đặc trưng có hệ số dương lớn nhất cho mỗi chủ đề từ `best` → dict `top_terms` (chủ đề → danh sách 5 n-gram). Có bao nhiêu trong số đó là **cặp âm tiết**?

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
vocab = best[0].get_feature_names_out()
clf = best[-1]
top_terms = {topic: [vocab[j] for j in np.argsort(-clf.coef_[i])[:5]] for i, topic in enumerate(clf.classes_)}
### END CODE HERE ###
for topic, terms in top_terms.items():
    print(f"{topic:10s}: {terms}")
share_bigrams = np.mean([" " in t for terms in top_terms.values() for t in terms])
print(f"tỉ lệ cặp âm tiết trong top đặc trưng: {share_bigrams:.0%}")

In [ ]:
check_equal(sorted(top_terms), sorted(train.topic.unique()), "đủ 6 chủ đề")
check_greater(share_bigrams, 0.2, "một phần đáng kể đặc trưng quan trọng là từ ghép hai âm tiết")

Quan sát: khoảng 1/4 đặc trưng mạnh nhất là cặp âm tiết, và nhiều âm tiết đơn quan trọng thực chất là **một nửa** của từ ghép ("sạn" ← "khách sạn", "ngân" ← "ngân hàng", "triển" ← "phát triển"). Một chi tiết dễ bỏ sót: `token_pattern` mặc định của scikit-learn (`\b\w\w+\b`) **bỏ các âm tiết một chữ cái** — "Bộ Y tế" thành cặp "bộ tế". Với tiếng Việt nên đặt `token_pattern=r"(?u)\b\w+\b"` (ở dữ liệu này metric gần như không đổi, nhưng đặc trưng dễ đọc hơn).

## Bước 7 (mở rộng) — Embedding kiểu fastText với PyTorch

Mô hình: mỗi âm tiết và cặp âm tiết có một vector 32 chiều; vector của tin = trung bình các vector (`nn.EmbeddingBag`); một lớp tuyến tính ra 6 chủ đề. Đây là ý tưởng của fastText (Joulin et al., 2016) — nhanh, mạnh cho phân loại văn bản. Ô dưới tự bỏ qua nếu chưa cài PyTorch.

In [ ]:
try:
    import torch
    from torch import nn
except ImportError:
    torch = None
    print('Chưa cài PyTorch — bỏ qua phần mở rộng: pip install -e ".[dl]"')

if torch is not None:
    torch.manual_seed(0)
    torch.set_num_threads(1)
    analyzer = CountVectorizer(ngram_range=(1, 2), min_df=2).fit(train.text)
    vocab_index = analyzer.vocabulary_
    tokenize = analyzer.build_analyzer()
    classes = sorted(train.topic.unique())

    def encode(texts):
        ids, offsets = [], [0]
        for t in texts:
            tok = [vocab_index[w] for w in tokenize(t) if w in vocab_index] or [0]
            ids.extend(tok)
            offsets.append(offsets[-1] + len(tok))
        return torch.tensor(ids), torch.tensor(offsets[:-1])

    embed, head = nn.EmbeddingBag(len(vocab_index), 32, mode="mean"), nn.Linear(32, len(classes))
    opt = torch.optim.Adam(list(embed.parameters()) + list(head.parameters()), lr=0.02)
    y_train = torch.tensor([classes.index(t) for t in train.topic])
    for epoch in range(15):
        perm = torch.randperm(len(train))
        for start in range(0, len(train), 64):
            idx = perm[start:start + 64]
            ids, offsets = encode(train.text.values[idx.numpy()])
            loss = nn.functional.cross_entropy(head(embed(ids, offsets)), y_train[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
    with torch.no_grad():
        ids, offsets = encode(test.text.values)
        pred_ft = [classes[i] for i in head(embed(ids, offsets)).argmax(1).tolist()]
    print(f"fastText-style macro-F1 test: {f1_score(test.topic, pred_ft, average='macro'):.3f} (TF-IDF + logistic: {f1_test:.3f})")

Trên bài toán phân loại **chủ đề** với vài nghìn tin, mô hình embedding đơn giản ngang ngửa TF-IDF + logistic. Mô hình ngữ cảnh lớn (PhoBERT — chương 31–32) có ích hơn khi nhãn phụ thuộc **ngữ nghĩa tinh tế** (mỉa mai, phủ định, cảm xúc) hoặc khi có rất ít dữ liệu gắn nhãn; cái giá là chậm hơn nhiều lần và cần GPU để fine-tune.

## Bước 8 (mở rộng) — Transformer encoder nhỏ, học từ đầu

Kiến trúc của [chương 31](../../../notebooks/31_transformers/README.md) áp dụng cho phân loại: mỗi âm tiết một vector **cộng vector vị trí**; 2 lớp **self-attention** cho mỗi âm tiết "nhìn" mọi âm tiết khác trong tin; lấy **trung bình** các vector đầu ra của những âm tiết thật (bỏ phần đệm `PAD`) → lớp tuyến tính ra 6 chủ đề.

**Bài tập:** hoàn thành `forward` — tính `pooled` là trung bình các vector `h` của token thật. `pad` là mask `(batch, độ dài)`, `True` ở vị trí đệm; chia cho **số token thật** của từng tin, không phải độ dài đã đệm.

In [ ]:
if torch is not None:
    import os

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.set_num_threads(max(1, (os.cpu_count() or 2) // 2))
    MAX_LEN = 64                                                        # tin dài nhất trong train ~55 âm tiết
    token_analyzer = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
    syllables = token_analyzer.build_analyzer()
    stoi = {w: i + 2 for i, w in enumerate(sorted(CountVectorizer(token_pattern=r"(?u)\b\w+\b", min_df=2).fit(train.text).vocabulary_))}

    def to_ids(texts):                                                  # 0 = PAD, 1 = âm tiết lạ
        X = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
        for i, t in enumerate(texts):
            ids = [stoi.get(w, 1) for w in syllables(t)][:MAX_LEN]
            X[i, :len(ids)] = ids
        return torch.tensor(X)

    class TinyTransformer(nn.Module):
        def __init__(self, vocab_size, n_classes, d_model=64, n_heads=4, n_layers=2):
            super().__init__()
            self.tok = nn.Embedding(vocab_size, d_model, padding_idx=0)
            self.pos = nn.Embedding(MAX_LEN, d_model)
            layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=2 * d_model, dropout=0.1, batch_first=True)
            self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
            self.head = nn.Linear(d_model, n_classes)

        def forward(self, x):
            pad = x == 0
            h = self.tok(x) + self.pos(torch.arange(x.shape[1], device=x.device))
            h = self.encoder(h, src_key_padding_mask=pad)               # attention bỏ qua vị trí PAD
            ### START CODE HERE ### (≈ 2 dòng)
            # STARTER: pooled = None    # (batch, d_model): trung bình h trên các token THẬT
            keep = (~pad).unsqueeze(-1).float()
            pooled = (h * keep).sum(1) / keep.sum(1).clamp(min=1)
            ### END CODE HERE ###
            return self.head(pooled)

    torch.manual_seed(0)
    transformer = TinyTransformer(len(stoi) + 2, len(classes)).to(device)
    optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-3)
    X_tr_ids, y_tr_ids = to_ids(train.text.values), torch.tensor([classes.index(t) for t in train.topic])
    for epoch in range(5):
        transformer.train()
        perm = torch.randperm(len(X_tr_ids))
        for start in range(0, len(perm), 64):
            idx = perm[start:start + 64]
            loss = nn.functional.cross_entropy(transformer(X_tr_ids[idx].to(device)), y_tr_ids[idx].to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    transformer.eval()
    with torch.no_grad():
        pred_tf_test = [classes[i] for i in transformer(to_ids(test.text.values).to(device)).argmax(1).tolist()]
        pred_tf_hard = [classes[i] for i in transformer(to_ids(hard.text.values).to(device)).argmax(1).tolist()]
        ids, offsets = encode(hard.text.values)                         # mô hình fastText-style của bước 7
        pred_ft_hard = [classes[i] for i in head(embed(ids, offsets)).argmax(1).tolist()]
    f1_transformer_test = f1_score(test.topic, pred_tf_test, average="macro")
    f1_transformer_hard = f1_score(hard.topic, pred_tf_hard, average="macro")
    print(f"{sum(p.numel() for p in transformer.parameters()):,} tham số | thiết bị: {device}")
    print(pd.DataFrame({"macro-F1 test": {"TF-IDF + logistic": f1_test, "fastText-style": f1_score(test.topic, pred_ft, average="macro"),
                                          "Transformer nhỏ (học từ đầu)": f1_transformer_test},
                        "macro-F1 tin khó": {"TF-IDF + logistic": f1_hard, "fastText-style": f1_score(hard.topic, pred_ft_hard, average="macro"),
                                             "Transformer nhỏ (học từ đầu)": f1_transformer_hard}}).round(3).to_string())

In [ ]:
if torch is not None:
    check_greater(f1_transformer_test, 0.9, "Transformer nhỏ: macro-F1 test > 0.9")
    check_greater(f1_transformer_hard, 0.88, "Transformer nhỏ: bền với tin nhiều câu gây nhiễu")

Transformer nhỏ học từ đầu trên 3.000 tin đạt mức **ngang** TF-IDF + logistic — không hơn, dù phức tạp và chậm hơn nhiều. Phân loại **chủ đề** chủ yếu dựa vào sự có mặt của từ khoá, thứ mà túi từ đã nắm tốt; attention chỉ thật sự có giá trị khi thứ tự và ngữ cảnh quyết định nghĩa (phủ định, mỉa mai, quan hệ giữa các thực thể). Sức mạnh thực tế của Transformer đến từ **tiền huấn luyện** trên kho văn bản khổng lồ rồi fine-tune — với tiếng Việt: PhoBERT, ViT5 (tham khảo, cần tải mô hình vài trăm MB):

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification   # pip install transformers
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")              # PhoBERT cần văn bản đã tách từ
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=6)
# → fine-tune vài epoch (Trainer hoặc vòng lặp PyTorch), learning rate nhỏ (2e-5)
```

## Bước 9 — Báo cáo

Viết 5–8 câu cho trưởng ban biên tập: độ chính xác, các lỗi chủ yếu và nguồn gốc (nhãn sai, tin đa chủ đề), đề xuất quy trình (mô hình gợi ý + biên tập viên xác nhận khi độ tin cậy thấp), giám sát (chủ đề mới, từ mới), bước tiếp theo.

✍️ Báo cáo:

**Báo cáo mẫu:** Mô hình TF-IDF (âm tiết + cặp âm tiết) với hồi quy logistic gắn đúng chủ đề ~96% tin trên tập kiểm tra (macro-F1 ~0.96), huấn luyện dưới một giây và giải thích được bằng các từ khoá đặc trưng của từng chủ đề. Hai hướng phức tạp hơn — embedding kiểu fastText và một Transformer nhỏ học từ đầu — không vượt được nó (macro-F1 ~0.95) nên chưa đáng chi phí vận hành; nếu sau này cần phân biệt sắc thái (tin đa chủ đề, phủ định), bước thử tiếp theo là fine-tune mô hình tiền huấn luyện như PhoBERT. Khoảng một nửa số "lỗi" thực ra là tin bị gắn nhãn sai trong dữ liệu gốc; phần còn lại là tin nói về nhiều chủ đề. Đề xuất: mô hình tự gắn chủ đề khi độ tin cậy cao (> 0.9), chuyển tin độ tin cậy thấp cho biên tập viên; đồng thời đưa các tin mô hình trái ý nhãn cũ đi rà soát để làm sạch dữ liệu. Đường cong học cho thấy thêm dữ liệu vẫn còn giúp — tiếp tục thu thập nhãn từ quy trình xác nhận. Giám sát: tỉ lệ tin độ tin cậy thấp và từ mới xuất hiện (chủ đề mới, sự kiện mới) — huấn luyện lại hằng tháng.